# Notebook 05: Model Evaluation

Evaluate all 3 models using Precision@K, Recall@K, NDCG@K, and Coverage.


## 1: Load Models and Data

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

# Models
hybrid_model = joblib.load('../trained_models/hybrid_model.pkl')
cb_model = joblib.load('../trained_models/content_based_model.pkl')
cf_model = joblib.load('../trained_models/collaborative_model.pkl')

# Feature artifacts
event_features = joblib.load('../trained_models/event_features.pkl')
user_profiles = joblib.load('../trained_models/user_profiles.pkl')
interaction_matrix = joblib.load('../trained_models/interaction_matrix.pkl')
event_id_to_idx = joblib.load('../trained_models/event_id_to_idx.pkl')
idx_to_event_id = joblib.load('../trained_models/idx_to_event_id.pkl')
user_id_to_idx = joblib.load('../trained_models/user_id_to_idx.pkl')
idx_to_user_id = joblib.load('../trained_models/idx_to_user_id.pkl')
user_interaction_counts = joblib.load('../trained_models/user_interaction_counts.pkl')

# Data
events = pd.read_csv('../data/events_clean.csv')
registrations = pd.read_csv('../data/registrations_clean.csv')

print(f'Models loaded. Events: {len(events)}, Users: {len(user_profiles)}')

## 2: Temporal Train/Test Split

In [ ]:
registrations['registration_date'] = pd.to_datetime(registrations['registration_date'])
split_date = registrations['registration_date'].quantile(0.8)

train_regs = registrations[registrations['registration_date'] <= split_date]
test_regs = registrations[registrations['registration_date'] > split_date]

test_ground_truth = test_regs.groupby('user_id')['event_id'].apply(set).to_dict()
train_user_events = train_regs.groupby('user_id')['event_id'].apply(set).to_dict()

print(f'Train: {len(train_regs)}, Test: {len(test_regs)}')
print(f'Test users: {len(test_ground_truth)}')

## 3: Evaluate All 3 Models

In [ ]:
from app.utils.metrics import evaluate_model

K_VALUES = [5, 10]
n_events = len(events)

# Content-Based prediction function
def cb_predict(user_id):
    profile = user_profiles.get(user_id)
    if profile is None:
        return []
    exclude = [event_id_to_idx[e] for e in train_user_events.get(user_id, set()) if e in event_id_to_idx]
    return cb_model.predict_for_user(profile, n=max(K_VALUES), exclude_ids=exclude)

# Collaborative prediction function
def cf_predict(user_id):
    uidx = user_id_to_idx.get(user_id)
    if uidx is None:
        return []
    exclude = [event_id_to_idx[e] for e in train_user_events.get(user_id, set()) if e in event_id_to_idx]
    return cf_model.predict(uidx, n=max(K_VALUES), exclude_ids=exclude)

# Hybrid prediction function
def hybrid_predict(user_id):
    profile = user_profiles.get(user_id)
    uidx = user_id_to_idx.get(user_id)
    cnt = user_interaction_counts.get(user_id, 0)
    if profile is None or uidx is None:
        return []
    exclude = [event_id_to_idx[e] for e in train_user_events.get(user_id, set()) if e in event_id_to_idx]
    return hybrid_model.predict(profile, uidx, cnt, n=max(K_VALUES), exclude_ids=exclude)

# Run evaluation
cb_metrics = evaluate_model('Content-Based', cb_predict, test_ground_truth, event_id_to_idx, idx_to_event_id, n_events, K_VALUES)
cf_metrics = evaluate_model('Collaborative', cf_predict, test_ground_truth, event_id_to_idx, idx_to_event_id, n_events, K_VALUES)
hy_metrics = evaluate_model('Hybrid', hybrid_predict, test_ground_truth, event_id_to_idx, idx_to_event_id, n_events, K_VALUES)

print('Evaluation complete')

## 4: Results Table

In [ ]:
all_metrics = {
    'Content-Based': cb_metrics,
    'Collaborative': cf_metrics,
    'Hybrid': hy_metrics,
}

# Build results table
rows = []
for model_name, m in all_metrics.items():
    row = {'Model': model_name}
    for key, val in m.items():
        if key != 'n_evaluated':
            row[key] = f'{val:.4f}'
        else:
            row[key] = val
    rows.append(row)

results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

## 5: Visualize Results

In [ ]:
# Bar chart comparing models across metrics
metric_keys = [f'P@{k}' for k in K_VALUES] + [f'R@{k}' for k in K_VALUES] + [f'NDCG@{k}' for k in K_VALUES] + ['Coverage']
model_names = list(all_metrics.keys())
colors = ['steelblue', 'coral', 'seagreen']

fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(metric_keys))
width = 0.25

for i, (name, metrics) in enumerate(all_metrics.items()):
    vals = [float(metrics.get(k, 0)) for k in metric_keys]
    ax.bar(x + i * width, vals, width, label=name, color=colors[i])

ax.set_xticks(x + width)
ax.set_xticklabels(metric_keys, rotation=45)
ax.set_ylabel('Score')
ax.set_title('Model Comparison', fontweight='bold', fontsize=14)
ax.legend()
ax.set_ylim(0, 1.1)

# Add value labels on bars
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', fontsize=7, padding=2)

plt.tight_layout()
plt.savefig('../data/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Heatmap of all metrics
heat_data = []
for name, metrics in all_metrics.items():
    row = {k: float(metrics.get(k, 0)) for k in metric_keys}
    heat_data.append(row)

heat_df = pd.DataFrame(heat_data, index=model_names)

fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(heat_df, annot=True, fmt='.3f', cmap='YlGn', ax=ax, vmin=0, vmax=1)
ax.set_title('Model Performance Heatmap', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('../data/model_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 6: Per-User Analysis

In [ ]:
from app.utils.metrics import precision_at_k, recall_at_k, ndcg_at_k

# Compute per-user NDCG@10 for the hybrid model
per_user_ndcg = []
for user_id, true_events in test_ground_truth.items():
    true_indices = {event_id_to_idx[e] for e in true_events if e in event_id_to_idx}
    if not true_indices:
        continue
    recs = hybrid_predict(user_id)
    rec_indices = [idx for idx, _ in recs]
    score = ndcg_at_k(rec_indices, true_indices, 10)
    cnt = user_interaction_counts.get(user_id, 0)
    per_user_ndcg.append({'user_id': user_id, 'ndcg': score, 'n_interactions': cnt})

per_user_df = pd.DataFrame(per_user_ndcg)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of NDCG scores
axes[0].hist(per_user_df['ndcg'], bins=20, color='mediumpurple', edgecolor='white')
axes[0].axvline(per_user_df['ndcg'].mean(), color='red', linestyle='--',
                label=f'Mean: {per_user_df["ndcg"].mean():.3f}')
axes[0].set_title('Hybrid NDCG@10 Distribution', fontweight='bold')
axes[0].set_xlabel('NDCG@10')
axes[0].set_ylabel('Users')
axes[0].legend()

# NDCG vs interaction count
axes[1].scatter(per_user_df['n_interactions'], per_user_df['ndcg'],
                alpha=0.5, color='teal', s=30)
axes[1].set_title('NDCG@10 vs Interaction Count', fontweight='bold')
axes[1].set_xlabel('Number of Interactions')
axes[1].set_ylabel('NDCG@10')

plt.tight_layout()
plt.savefig('../data/per_user_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Users evaluated: {len(per_user_df)}')
print(f'Mean NDCG@10: {per_user_df["ndcg"].mean():.4f}')
print(f'Median NDCG@10: {per_user_df["ndcg"].median():.4f}')

## 7: Weight Sensitivity Analysis

Test different content/collab weight ratios for the hybrid model.

In [ ]:
from app.models.hybrid import HybridRecommender

weight_configs = [
    (1.0, 0.0),  # content only
    (0.8, 0.2),
    (0.6, 0.4),  # default
    (0.5, 0.5),
    (0.4, 0.6),
    (0.2, 0.8),
    (0.0, 1.0),  # collab only
]

weight_results = []
for cw, cfcw in weight_configs:
    h = HybridRecommender(content_weight=cw, collab_weight=cfcw)
    h.fit(event_features, interaction_matrix, n_factors=20)
    
    def predict_fn(uid, model=h):
        profile = user_profiles.get(uid)
        uidx = user_id_to_idx.get(uid)
        cnt = user_interaction_counts.get(uid, 0)
        if profile is None or uidx is None:
            return []
        exclude = [event_id_to_idx[e] for e in train_user_events.get(uid, set()) if e in event_id_to_idx]
        return model.predict(profile, uidx, cnt, n=10, exclude_ids=exclude)
    
    m = evaluate_model(f'H({cw}/{cfcw})', predict_fn, test_ground_truth,
                       event_id_to_idx, idx_to_event_id, len(events), [5, 10])
    m['content_w'] = cw
    m['collab_w'] = cfcw
    weight_results.append(m)

weight_df = pd.DataFrame(weight_results)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(weight_df['content_w'], weight_df['NDCG@10'], 'o-', color='teal', label='NDCG@10', linewidth=2)
ax.plot(weight_df['content_w'], weight_df['P@10'], 's--', color='coral', label='P@10', linewidth=2)
ax.plot(weight_df['content_w'], weight_df['R@10'], '^:', color='steelblue', label='R@10', linewidth=2)
ax.set_xlabel('Content Weight (1 - collab_weight)')
ax.set_ylabel('Score')
ax.set_title('Hybrid Weight Sensitivity', fontweight='bold', fontsize=14)
ax.legend()
ax.set_xlim(-0.05, 1.05)
plt.tight_layout()
plt.savefig('../data/weight_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

# Find and print best config
best_idx = weight_df['NDCG@10'].idxmax()
best = weight_df.iloc[best_idx]
print(f'Best config: content={best["content_w"]}, collab={best["collab_w"]}')
print(f'  NDCG@10: {best["NDCG@10"]:.4f}, P@10: {best["P@10"]:.4f}, R@10: {best["R@10"]:.4f}')